In [1]:
import numpy as np
from scipy.special import comb, erfc
import math

PI = np.pi

def delta(q: int, n: int, eta1: float, eta2: float, d0: int, d1: int, d2: int, p: int) -> float:
    V1 = eta1
    V2 = eta2

    Vb = (q ** 2) / ((1 << (2 * d0)) * 12.0)
    Vu = (q ** 2) / ((1 << (2 * d1)) * 12.0)
    Vv = (q ** 2) / ((1 << (2 * d2)) * 12.0)

    std_sq = (3 * n / 2.0) * (2 * V1 * V2 + Vb * V1 + Vu * V2) + V2 + Vv
    std = np.sqrt(std_sq)

    B = (q + p) // (2 * p)
    err = erfc(B / (np.sqrt(2.0) * std))
    return err


def Delta(l: int, t: int, delta_val: float) -> float:
    total = 0.0
    for j in range(t + 1, l + 1):
        c = comb(l, j, exact=False)
        total += c * (delta_val ** j) * ((1.0 - delta_val) ** (l - j))
    return total


def payload_size(n: int, l: int, d0: int, d1: int, d2: int):
    pk = 256 + d0 * n
    pk = pk // 8

    ct = d1 * n + d2 * l
    ct = ct // 8

    print(f"size of pk: {pk} Bytes")
    print(f"size of ct: {ct} Bytes")



In [2]:
def main():
    # Param Set: [q, n, eta1, eta2, d0, d1, d2, p, l, t]
    params = [
        [3457, 576, 2.0, 2.0, 10, 10, 5, 2, 523, 1],    # 128
        [3457, 864, 1.0, 2.0, 10, 11, 6, 2, 523, 1],    # 192
        [3457, 1152, 1.0, 1.0, 11, 10, 6, 2, 523, 1],   # 256
        [3457, 1728, 1.0, 1.0, 11, 11, 6, 2, 523, 1],   # 384
        [3457, 2304, 1.0, 1.0, 12, 12, 7, 2, 523, 1]    # 512
    ]

    for idx, p in enumerate(params):
        q, n, eta1, eta2, d0, d1, d2, p, l, t = p

        print(f"|  q={q}, n={n}  |  eta1={eta1}, eta2={eta2}  |  d0={d0}, d1={d1}, d2={d2}  |")

        payload_size(n, l, d0, d1, d2)

        delta_per_bit = delta(q, n, eta1, eta2, d0, d1, d2, p)
        log2_delta = math.log2(delta_per_bit)
        print(f"Error Rate per Coefficient = {delta_per_bit:.6e} (2^{log2_delta:.4f})")

        all_fail_rate = Delta(l, 0, delta_per_bit)
        log2_all = math.log2(all_fail_rate)
        print(f"Decryption Failure Rate = {all_fail_rate:.6e} (2^{log2_all:.4f})")

        # (256, 247, 4) - ECC
        ecc_rate1 = Delta(l, 1, delta_per_bit)  # Correctable
        ecc_rate2 = Delta(l, 2, delta_per_bit)  # Detectable
        log2_e1 = math.log2(ecc_rate1)
        log2_e2 = math.log2(ecc_rate2)
        print(f"ECC Error Rate <= {ecc_rate1:.6e} (2^{log2_e1:.4f}) ~ {ecc_rate2:.6e} (2^{log2_e2:.4f})\n")


if __name__ == "__main__":
    main()

|  q=3457, n=576  |  eta1=2.0, eta2=2.0  |  d0=10, d1=10, d2=5  |
size of pk: 752 Bytes
size of ct: 1046 Bytes
Error Rate per Coefficient = 2.949624e-16 (2^-51.5903)
Decryption Failure Rate = 1.542653e-13 (2^-42.5597)
ECC Error Rate <= 1.187615e-26 (2^-86.1221) ~ 6.083573e-40 (2^-130.2722)

|  q=3457, n=864  |  eta1=1.0, eta2=2.0  |  d0=10, d1=11, d2=6  |
size of pk: 1112 Bytes
size of ct: 1580 Bytes
Error Rate per Coefficient = 4.092597e-24 (2^-77.6933)
Decryption Failure Rate = 2.140428e-21 (2^-68.6626)
ECC Error Rate <= 2.286337e-42 (2^-138.3279) ~ 1.625009e-63 (2^-208.5810)

|  q=3457, n=1152  |  eta1=1.0, eta2=1.0  |  d0=11, d1=10, d2=6  |
size of pk: 1616 Bytes
size of ct: 1832 Bytes
Error Rate per Coefficient = 4.558796e-30 (2^-97.4692)
Decryption Failure Rate = 2.384250e-27 (2^-88.4385)
ECC Error Rate <= 2.836890e-54 (2^-177.8798) ~ 2.245997e-81 (2^-267.9088)

|  q=3457, n=1728  |  eta1=1.0, eta2=1.0  |  d0=11, d1=11, d2=6  |
size of pk: 2408 Bytes
size of ct: 2768 Bytes
Error 